# GPU Mesh Depth Animation Test

This notebook is an isolated test lane for estimating animated mesh depth without Blender. It keeps the Meshy smart rig and the authentic 12 fps idle → move-start → walk sequence explicit. Idle is first because later motion has to seed from rest, not from the imported bind. The current implementation uses the GPU-resident skinned geometry path from `roc_gpu_cascade_experiment.py` and the commit-pinned `nvdiffrast` CUDA extension in `third_party/gpu_pose_v2`. It fails closed rather than silently substituting Blender or a monocular depth model.

The comparison target is VDA-S depth, not RIFE and not DA3. Idle is a posing prefix only; delivery still slices it out.

In [ ]:
from pathlib import Path
import sys
ROOT = Path(r'V:/Games/Olden Era Playtest Tweak')
TOOL = ROOT / 'tools/homm3_silhouette_pose_fit'
THIRD_PARTY = TOOL / 'third_party' / 'gpu_pose_v2'
for _path in (THIRD_PARTY, TOOL):
    if str(_path) not in sys.path:
        sys.path.insert(0, str(_path))
import torch
from roc_gpu_cascade_experiment import capability_report
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print(capability_report())
if not torch.cuda.is_available():
    raise RuntimeError('This notebook requires CUDA.')
if not capability_report().get('nvdiffrast'):
    raise RuntimeError('nvdiffrast is required for exact triangle depth. Run setup_gpu_pose_v2.ps1, then restart the kernel.')

In [ ]:
# Tunable experiment configuration. Keep this cell as the only notebook setup surface.
# Meshy smart-rig remains the authoritative skeleton and skin source.
UNIT = 'h3_crusader'
MESH = ROOT / 'artifacts/meshy_downloads/h3_crusader_smartrigged.glb'
IDLE_START_MOVE = ROOT / 'artifacts/pose_picker_runs/h3_crusader/idle_start_move'
MOVEMENT_SOURCE = IDLE_START_MOVE / 'frames_def12_nn_idle_start_move_shadow_culled'
IDLE_SOURCE = MOVEMENT_SOURCE  # frame 00 is idle rest; do not point at isolated idle_g2
SEED_CAMERA_FIT = IDLE_START_MOVE / 'roc_sprite_camera_fit.json'
PROFILE_OUTPUT = IDLE_START_MOVE / 'profile_registration'
PROFILE_FRAME_INDEX = 0  # idle frame 00 of the concatenated sequence registers the imported mesh
PROFILE_YAW_STEP_DEG = 15.0  # static camera search granularity; not an animation joint step
PROFILE_CAMERA_BASIS_MODE = 'upright_turntable'  # proven idle-silhouette orientation; geometry_broad_thin remains an optional ablation
PROFILE_ROLL_RANGE_DEG = 0.0  # keep the imported rig upright; expose roll only for a deliberate ablation
PROFILE_ROLL_STEP_DEG = 1.0
USE_PROFILE_REGISTRATION = True
CAMERA_FIT = SEED_CAMERA_FIT  # replaced by the registration cell below
VDA_DEPTH = IDLE_START_MOVE / 'frames_def12_nn_idle_start_move_depth_vda'
DEPTH_WEIGHT = 0.25  # normalized depth loss weight; keep below 0.5 while registration is being validated
OUTPUT = ROOT / 'artifacts/depth_model_comparison/gpu_raster_depth_animation' / UNIT
RASTER_WIDTH, RASTER_HEIGHT = 192, 153  # preserve the selected DEF frame aspect ratio in every GPU preview
BLENDER_RENDER_RESOLUTION = (RASTER_WIDTH, RASTER_HEIGHT)  # parity preview uses the exact same canvas; upscale separately for delivery
SKIN_LEVEL = 1  # weighted LBS; 0 selects the dominant joint only
MESH_SCALE = 1.0  # shared by the GPU bind preview and final Blender export
SHOW_VDA_COMPARISON = True  # display the high-quality VDA depth reference below the solved depth
FRAME_INDICES = tuple(range(18))  # idle 0-7, move-start 8-9, walk 10-17; never RIFE interpolation targets

# Root motion is animation, not profile registration.  Keep the first test
# conservative: the profile cell already solved static facing, so the root may
# roll in the image plane and translate, but cannot silently become a second
# camera that hides a failed joint solution.  Add camera_right/camera_up only
# for an action that visibly requires out-of-plane body rotation.
ROOT_SEARCH_ENABLED = True  # seed each selected frame with a broad rigid-motion search
ROOT_HYPOTHESES = 64  # exact filled-triangle candidates retained by the root search
ROOT_ROTATION_AXES = ('camera_forward',)  # in-plane roll only for this registration-safe baseline
ROOT_TRANSLATION_AXES = ('lateral', 'vertical')  # image-space body shift; no hidden depth drift
ROOT_ROTATION_LIMIT_DEG = 35.0  # broad enough for action roll, narrow enough to reject upside-down minima
ROOT_ROTATION_LIMIT_OPTIMIZATION_DEG = 35.0  # continuous optimization bound for the same root channel
ROOT_TRANSLATION_RADIUS_BODY_LENGTHS = 0.20  # initial rigid translation search radius
ROOT_TRANSLATION_LIMIT_BODY_LENGTHS = 0.25  # optimization bound for the rigid translation
ROOT_ITERATIONS = 24  # root-only refinement iterations
ARTICULATED_ITERATIONS = 80  # all-bone refinement iterations after root registration
FINE_ITERATIONS = 30  # small final bone-only refinement pass
BASE_LEARNING_RATE = 0.01  # lower than the old proxy-tuned value for filled-triangle edges
FINE_LEARNING_RATE = 0.003  # final small-step rate; rejected steps still trigger bounded escape increases
FREEZE_ROOT_AFTER_ROOT_STAGE = True  # do not let joints pull static registration into a local minimum
FINE_ROOT_ENABLED = False  # keep final refinement focused on joints, not camera-like root motion
EXACT_RASTER_ENABLED = True  # nvdiffrast filled triangles with antialiased edge gradients

## What this test must prove

1. The GLB vertices and triangle indices stay resident on CUDA.
2. Weighted skinning and projection happen without Blender.
3. The depth image comes from rasterized mesh triangles, not sparse points or a vision model.
4. The VDA-S comparison uses the same idle → start → walk frames as the GPU solve (18 authentic 12 fps plates). Isolated walk without idle is not this test.

The exact raster backend is intentionally a hard gate. If `nvdiffrast` is unavailable, this notebook stops at preflight instead of presenting the existing soft point proxy as a triangle depth result.

In [ ]:
source_frames = sorted(MOVEMENT_SOURCE.glob('*.png'), key=lambda p: int(p.stem))
idle_frames = sorted(IDLE_SOURCE.glob('*.png'), key=lambda p: int(p.stem))
vda_frames = sorted(VDA_DEPTH.glob('*.png'), key=lambda p: int(p.stem))
assert len(source_frames) == len(FRAME_INDICES), (len(source_frames), FRAME_INDICES)
assert len(vda_frames) == len(source_frames), (len(vda_frames), len(source_frames))
assert idle_frames and PROFILE_FRAME_INDEX < len(idle_frames), (IDLE_SOURCE, PROFILE_FRAME_INDEX)
assert MESH.is_file(), MESH
assert SEED_CAMERA_FIT.is_file(), SEED_CAMERA_FIT
from materialize_rife_source_frames import require_idle_leading_source_manifest
_idle_groups = require_idle_leading_source_manifest(IDLE_START_MOVE / 'frames_def12_nn_idle_start_move' / 'def_frames.json')
assert _idle_groups == (2, 20, 0), _idle_groups
print(f'{UNIT}: {len(source_frames)} authentic idle-leading frames (idle, start, walk); registration uses concatenated frame {PROFILE_FRAME_INDEX:02d}')
print('mesh:', MESH)
print('seed camera:', SEED_CAMERA_FIT)
print('VDA reference:', VDA_DEPTH)

In [ ]:
# Static registration is solved once from concatenated idle frame 00, not from a walk stride.
# Registration has two separate authorities: idle/bind geometry chooses orientation,
# while later concatenated frames supply the action. The same folder holds both
# because idle is a prefix of idle_start_move, not a sibling isolated clip.
from gpu_profile_camera_initializer import ProfileCameraInitConfig, initialize_profile_camera
from PIL import Image
if USE_PROFILE_REGISTRATION:
    PROFILE_CONFIG = ProfileCameraInitConfig(
        resolution=(192, 153),
        yaw_step_deg=PROFILE_YAW_STEP_DEG,
        roll_range_deg=PROFILE_ROLL_RANGE_DEG,
        roll_step_deg=PROFILE_ROLL_STEP_DEG,
        camera_basis_mode=PROFILE_CAMERA_BASIS_MODE,
        top_k=8,
    )
    PROFILE_RESULT = initialize_profile_camera(
        src_glb=MESH,
        seed_camera_fit=SEED_CAMERA_FIT,
        target_frame_dir=IDLE_SOURCE,
        fit_target_frame_dir=MOVEMENT_SOURCE,
        frame_index=PROFILE_FRAME_INDEX,
        output_dir=PROFILE_OUTPUT,
        config=PROFILE_CONFIG,
        progress=True,
    )
    CAMERA_FIT = PROFILE_RESULT.camera_fit_path
    print('registered camera:', CAMERA_FIT)
    print('registration report:', PROFILE_RESULT.report_path)
    display(Image.open(PROFILE_RESULT.preview_path))
else:
    CAMERA_FIT = SEED_CAMERA_FIT
    print('profile registration disabled; using seed camera:', CAMERA_FIT)

In [ ]:
# The exact triangle renderer and CUDA buffers are owned by the source module.
# This import is deliberately deferred until after the hard preflight above.
from gpu_raster_depth_animation import prepare_gpu_raster_depth, render_gpu_depth_animation
rig = prepare_gpu_raster_depth(
    mesh_path=MESH,
    camera_fit=CAMERA_FIT,
    width=RASTER_WIDTH,
    height=RASTER_HEIGHT,
    mesh_scale=MESH_SCALE,
    skin_level=SKIN_LEVEL,
)
print(rig.summary())

In [ ]:
result = render_gpu_depth_animation(
    rig,
    frame_indices=FRAME_INDICES,
    output_dir=OUTPUT,
    target_vda_depth_dir=VDA_DEPTH if SHOW_VDA_COMPARISON else None,
)
print(result.manifest_path)
display(result.contact_sheet())
print('bind exact GPU triangle alpha:', result.alpha_gif_path)
display(Image.open(result.alpha_gif_path))

## Interpretation

The GPU depth is a geometric consequence of the selected mesh, bind skeleton, skin weights, pose, and camera. It is not expected to reproduce VDA's hidden-surface guesses automatically. The useful test is whether the rasterized mesh depth changes coherently with articulation and whether its visible-side ordering agrees with VDA enough to guide the solver.

## Animation solve and real render
The previous cell is only the bind-pose CUDA smoke test. This cell runs the actual selected-frame pose solve, exports exact CUDA triangle depth for the solved poses, and then invokes Blender only as the final lit-render/parity step. VDA depth is near-bright; this path does not reuse the old reversed preview.

In [ ]:
from roc_gpu_cascade_experiment import GpuCascadeConfig, export_gpu_render_preview
from gpu_raster_depth_animation import solve_gpu_depth_animation

# CAMERA_FIT is the output of the explicit static-registration cell.
# Do not silently replace it with the old movement-folder seed here.
if not CAMERA_FIT.is_file():
    raise FileNotFoundError(CAMERA_FIT)

SOLVE_OUTPUT = OUTPUT / 'animation_solver'
SOLVER_CONFIG = GpuCascadeConfig(
    device='cuda',
    resolution=(RASTER_WIDTH, RASTER_HEIGHT),
    root_search_enabled=ROOT_SEARCH_ENABLED,
    root_hypotheses=ROOT_HYPOTHESES,
    root_rotation_axes=ROOT_ROTATION_AXES,
    root_translation_axes=ROOT_TRANSLATION_AXES,
    root_rotation_limit_deg=ROOT_ROTATION_LIMIT_DEG,
    root_rotation_limit_optimization_deg=ROOT_ROTATION_LIMIT_OPTIMIZATION_DEG,
    root_translation_radius_body_lengths=ROOT_TRANSLATION_RADIUS_BODY_LENGTHS,
    root_translation_limit_body_lengths=ROOT_TRANSLATION_LIMIT_BODY_LENGTHS,
    optimize_root_translation=True,
    optimize_root_rotation=True,
    optimize_bones=True,
    root_iterations=ROOT_ITERATIONS,
    articulated_iterations=ARTICULATED_ITERATIONS,
    fine_iterations=FINE_ITERATIONS,
    learning_rate=BASE_LEARNING_RATE,
    fine_learning_rate=FINE_LEARNING_RATE,
    freeze_root_after_root_stage=FREEZE_ROOT_AFTER_ROOT_STAGE,
    fine_root_enabled=FINE_ROOT_ENABLED,
    exact_raster_enabled=EXACT_RASTER_ENABLED,
    depth_target_dir=VDA_DEPTH,
    depth_weight=DEPTH_WEIGHT,
    return_masks=True,
)

# The GPU optimizer scores the same filled triangles that the exact alpha preview displays.
# VDA is retained as a configurable depth reference and is compared in the next cell.
SOLVE_RESULT = solve_gpu_depth_animation(
    src_glb=MESH,
    camera_fit=CAMERA_FIT,
    target_frame_dir=MOVEMENT_SOURCE,
    frame_indices=FRAME_INDICES,
    config=SOLVER_CONFIG,
    output_dir=SOLVE_OUTPUT,
    render_blender=False,
)
print('solved score:', SOLVE_RESULT.pose_result['score'])
print('exact CUDA depth GIF:', SOLVE_RESULT.depth_gif)

In [ ]:
# Compare the solved GPU depth convention to VDA and produce the actual lit mesh preview.
import numpy as np
from PIL import Image
gpu_depth = np.stack([np.asarray(Image.open(SOLVE_RESULT.depth_dir / f'{int(f):02d}.png').convert('L'), dtype=np.float32) / 255.0 for f in FRAME_INDICES])
vda_depth = np.stack([np.asarray(Image.open(VDA_DEPTH / f'{int(f):02d}.png').convert('L').resize((RASTER_WIDTH, RASTER_HEIGHT)), dtype=np.float32) / 255.0 for f in FRAME_INDICES])
print('mean VDA depth MAE:', float(np.abs(gpu_depth - vda_depth).mean()))
display(Image.open(SOLVE_RESULT.depth_gif))

# These are separate artifacts on purpose: exact GPU alpha is the solver silhouette,
# while the later GIF is the independent Blender lit-render/parity output.
from IPython.display import Image as NotebookImage, display as notebook_display
_gpu_alpha_value = SOLVE_RESULT.alpha_gif_path or SOLVE_RESULT.pose_result.get('gpuAlphaGif')
if _gpu_alpha_value and Path(_gpu_alpha_value).is_file():
    print('exact GPU triangle alpha:', _gpu_alpha_value)
    notebook_display(NotebookImage(filename=str(_gpu_alpha_value)))
else:
    raise FileNotFoundError(f'exact GPU alpha preview is missing: {_gpu_alpha_value}')

# Fill in the same selector-derived paths used by the normal GPU preview cell.
REAL_RENDER = export_gpu_render_preview(
    SOLVE_RESULT.pose_result,
    src_glb=MESH, unit_name=UNIT, animation_name='move_g0',
    pilot_dir=OUTPUT, camera_fit_path=CAMERA_FIT,
    frame_indices=FRAME_INDICES, output_dir=SOLVE_OUTPUT / 'blender_preview',
    source_fps=12.0, output_fps=60.0, mesh_scale=MESH_SCALE,
    render_resolution=BLENDER_RENDER_RESOLUTION,
)
_render_gif = Path(REAL_RENDER['gif'])
print('real Blender render:', _render_gif)
notebook_display(NotebookImage(filename=str(_render_gif)))

# Parity gate: compare the exact CUDA alpha and the corresponding Blender
# keyframe at the same low-resolution canvas.  This is intentionally
# separate from the solver score so a stretched or divergent export
# cannot masquerade as a successful fit.
_render_paths = [Path(value) for value in REAL_RENDER['framePaths']]
_key_stride = max(1, round(REAL_RENDER['outputFps'] / REAL_RENDER['sourceFps']))
_parity_rows = []
_first_gpu_alpha = None
_first_blender_alpha = None
for _position, _frame in enumerate(FRAME_INDICES):
    _gpu_alpha = np.asarray(Image.open(SOLVE_RESULT.alpha_dir / f'{int(_frame):02d}.png').convert('L'), dtype=np.uint8) > 8
    _blender_alpha = np.asarray(Image.open(_render_paths[_position * _key_stride]).convert('RGBA').getchannel('A').resize((RASTER_WIDTH, RASTER_HEIGHT), Image.Resampling.NEAREST), dtype=np.uint8) > 8
    if _position == 0:
        _first_gpu_alpha, _first_blender_alpha = _gpu_alpha, _blender_alpha
    _union = _gpu_alpha | _blender_alpha
    _parity_rows.append(float((_gpu_alpha & _blender_alpha).sum() / max(1, _union.sum())))
print('GPU alpha vs Blender keyframe IoU:', _parity_rows, 'mean=', float(np.mean(_parity_rows)), 'min=', float(np.min(_parity_rows)))
_overlay = np.zeros((RASTER_HEIGHT, RASTER_WIDTH, 3), dtype=np.uint8)
_overlay[..., 0] = np.where(_first_gpu_alpha & ~_first_blender_alpha, 255, 0)
_overlay[..., 1] = np.where(_first_blender_alpha & ~_first_gpu_alpha, 255, 0)
_overlay[..., 2] = np.where(_first_gpu_alpha & _first_blender_alpha, 255, 0)
print('frame 00 parity overlay: red=GPU-only, green=Blender-only, blue=both')
display(Image.fromarray(_overlay, mode='RGB').resize((RASTER_WIDTH * 3, RASTER_HEIGHT * 3), Image.Resampling.NEAREST))
print('real Blender MP4:', REAL_RENDER.get('mp4'))